In [ ]:
from openai import OpenAI
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
import gradio as gr


from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

### Load Documents

In [7]:
loader = DirectoryLoader(
    "./documents",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(len(documents))

3


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

In [ ]:
embedding_model = OpenAIEmbeddings(
    openai_api_key="OPENROUTER_API_KEY",
    openai_api_base="OPENROUTER_API_BASE_URL",
)

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

### Store in Vector Database

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./vectordb"
)

### Query the Database

In [ ]:
query = "What is the leave policy?"

results = vector_db.similarity_search(query, k=3)

results

In [ ]:
def ask_question(question):

    # Retrieve relevant documents
    results = vector_db.similarity_search(question, k=3)

    # Combine context
    context = "\n".join([doc.page_content for doc in results])

    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )

    answer = response.choices[0].message.content

    return answer

In [ ]:
interface = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(label="Ask a question"),
    outputs=gr.Textbox(label="Answer"),
    title="📚 HR Policy Assistant",
    description="This chatbot answers questions based on company documents using Retrieval-Augmented Generation (RAG).",
    theme="soft"
)

In [ ]:
interface.launch()